> **AI-assisted:** Review paths before running.

# Build DA Metadata GeoJSON

This Colab notebook is designed for the case where your code repository stays on your local computer and only the raw data lives in Google Drive.

You only need to place two support files in Google Drive:

- `build_da_metadata_geojson.py`
- `fed_boundaries_2023.geojson`

Then Colab will read the raw CRMP data from Drive, generate all `da_<FED_NUM>.geojson` files into a Drive output folder, and also create a downloadable zip.

In [ ]:
# ==========================================
# Cell 1: Mount Google Drive and install dependencies
# ==========================================
try:
    from google.colab import drive  # pyright: ignore[reportMissingImports]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print('[WARN] This notebook is intended for Google Colab.')
else:
    drive.mount('/content/drive')
    get_ipython().run_line_magic('pip', 'install -q geopandas fiona shapely pandas pyogrio')  # pyright: ignore[reportUndefinedVariable]
    print('[OK] Google Drive mounted and geopandas dependencies installed')

In [ ]:
# ==========================================
# Cell 2: Configure Drive paths
# ==========================================
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/CSCC01 - Group Project/Data')
SCRIPT_PATH = DATA_ROOT / 'build_da_metadata_geojson.py'
FED_BOUNDARIES_GEOJSON = DATA_ROOT / 'fed_boundaries_2023.geojson'
OUTPUT_DIR = DATA_ROOT / 'colab_outputs' / 'da_metadata_geojson'
OUTPUT_ZIP = DATA_ROOT / 'colab_outputs' / 'da_metadata_geojson.zip'
PROVINCES = ['ab', 'bc', 'mb', 'nb', 'nl', 'ns', 'nt', 'nu', 'on', 'pe', 'qc', 'sk', 'yt']
CLEAN_EXISTING = True

RAW_ROOT_CANDIDATES = [
    DATA_ROOT / 'CRMP-full-data',
    DATA_ROOT / 'CRMP-full-data' / 'raw_data',
    DATA_ROOT / 'CRMP-full-data' / 'raw-data',
    DATA_ROOT / 'CRMP-full-data' / 'CRMP-full-data',
    DATA_ROOT / 'CRMP-full-data' / 'CRMP-full-data' / 'raw_data',
    DATA_ROOT / 'CRMP-full-data' / 'CRMP-full-data' / 'raw-data',
]


def looks_like_raw_root(path: Path) -> bool:
    return (path / 'statistics_canada' / 'census_boundaries').exists()


def resolve_raw_root(candidates):
    for candidate in candidates:
        candidate = candidate.expanduser()
        if looks_like_raw_root(candidate):
            return candidate.resolve()
    return None


RAW_ROOT = resolve_raw_root(RAW_ROOT_CANDIDATES)

required_paths = {
    'DATA_ROOT': DATA_ROOT,
    'SCRIPT_PATH': SCRIPT_PATH,
    'FED_BOUNDARIES_GEOJSON': FED_BOUNDARIES_GEOJSON,
}

for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f'{label} is missing: {path}')

if RAW_ROOT is None:
    print('[ERROR] Could not resolve RAW_ROOT. Checked:')
    for candidate in RAW_ROOT_CANDIDATES:
        print(f'  - {candidate}')
    raise FileNotFoundError('RAW_ROOT is missing or nested unexpectedly.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print(f'[OK] DATA_ROOT = {DATA_ROOT}')
print(f'[OK] RAW_ROOT = {RAW_ROOT}')
print(f'[OK] SCRIPT_PATH = {SCRIPT_PATH}')
print(f'[OK] FED_BOUNDARIES_GEOJSON = {FED_BOUNDARIES_GEOJSON}')
print(f'[OK] OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# ==========================================
# Cell 3: Import the standalone script from Drive and build files
# ==========================================
import importlib.util
import json
import sys

module_name = 'build_da_metadata_geojson_standalone'
spec = importlib.util.spec_from_file_location(module_name, SCRIPT_PATH)
module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = module
spec.loader.exec_module(module)

result = module.build_metadata_geojson(
    raw_root=RAW_ROOT,
    province_codes=PROVINCES,
    clean=CLEAN_EXISTING,
    fed_boundaries_geojson=FED_BOUNDARIES_GEOJSON,
    metadata_out_dir=OUTPUT_DIR,
)

print(json.dumps({
    'metadata_out_dir': result['metadata_out_dir'],
    'generated_file_count': len(result['summary']),
    'first_files': result['summary'][:10],
}, indent=2))

In [ ]:
# ==========================================
# Cell 4: Create a zip in Drive and download it
# ==========================================
from zipfile import ZipFile

with ZipFile(OUTPUT_ZIP, 'w') as archive:
    for path in sorted(OUTPUT_DIR.glob('da_*.geojson')):
        archive.write(path, arcname=path.name)

print(f'[OK] Created zip in Drive: {OUTPUT_ZIP}')
print(f'[OK] Generated files remain in Drive folder: {OUTPUT_DIR}')

if IN_COLAB:
    from google.colab import files  # pyright: ignore[reportMissingImports]
    files.download(str(OUTPUT_ZIP))